In [1]:
# ============================================================
# v23 REGISTER DIAGNOSTICS
# Three read-only diagnostics. Does not modify the model.
#   1. activation_map        -- which slots activate when?
#   2. failure_conditioned   -- which slot diverges on wrong answers?
#   3. slot_ablation         -- which slots are load-bearing?
# ============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import os, copy

REG_NAMES = ['ones_A', 'tens_A', 'ones_B', 'tens_B', 'carry', 'answer']
DIAG_DIR  = os.path.join(DRIVE_DIR, 'v23_diagnostics')
os.makedirs(DIAG_DIR, exist_ok=True)


# ── 1. ACTIVATION MAP ────────────────────────────────────────────────────────
def activation_map(model, stage, n_problems=50):
    """
    For n_problems random problems at this stage, record ||register_slot||
    at every timestep. Plot the average trace per slot over time.

    Reads: which slots wake up during which phases of the input.
    """
    R = REG_DIM
    model.eval()
    all_traces = []   # (n_problems, T, 6)
    with torch.no_grad():
        for _ in range(n_problems):
            text, a, b, ans, carry = make_problem(stage)
            ids = torch.tensor([TOK.encode(text)], dtype=torch.long, device=device)
            h = model.cell.init_h(1)
            slot_norms = []
            for t in range(ids.size(1) - 1):
                _, h = model.cell.step(ids[0, t:t+1], h)
                h = cap_registers(h, R)
                norms = [h[0, i*R:(i+1)*R].norm().item() for i in range(6)]
                slot_norms.append(norms)
            all_traces.append(np.array(slot_norms))

    # Pad / truncate to common length for averaging
    min_T = min(t.shape[0] for t in all_traces)
    arr = np.stack([t[:min_T] for t in all_traces])  # (n, T, 6)
    mean_trace = arr.mean(0)   # (T, 6)
    std_trace  = arr.std(0)

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for i in range(6):
        ax.plot(mean_trace[:, i], label=REG_NAMES[i], color=colors[i], lw=1.6)
        ax.fill_between(np.arange(min_T),
                        mean_trace[:, i] - std_trace[:, i],
                        mean_trace[:, i] + std_trace[:, i],
                        color=colors[i], alpha=0.15)
    ax.set_xlabel("timestep")
    ax.set_ylabel("‖register slot‖")
    ax.set_title(f"Per-register activation map | stage {stage} | n={n_problems}")
    ax.legend(loc='upper left', ncol=2, fontsize=9)
    ax.grid(alpha=0.3)
    out = os.path.join(DIAG_DIR, f'activation_map_stage{stage}.png')
    plt.tight_layout(); plt.savefig(out, dpi=130); plt.close()
    print(f"  saved {out}")

    # Print which slots are dead (always near zero)
    final_norms = mean_trace[-1]
    print(f"\n  Final-step register norms (stage {stage}):")
    for i, name in enumerate(REG_NAMES):
        flag = "  ← DEAD" if final_norms[i] < 0.01 else ""
        print(f"    {name:8s} {final_norms[i]:.4f}{flag}")
    return mean_trace


# ── 2. FAILURE-CONDITIONED STATES ────────────────────────────────────────────
def failure_conditioned(model, stage, n_problems=200):
    """
    Run eval. Split into correct vs incorrect. For each register slot,
    compare the hidden-state distribution between the two groups.

    Reads: which slot's representation differs between success/failure.
    """
    R = REG_DIM
    model.eval()
    correct_states = []   # list of (6, R) per problem
    wrong_states   = []
    correct_finals = []
    wrong_finals   = []

    with torch.no_grad():
        for _ in range(n_problems):
            text, a, b, true_ans, true_carry = make_problem(stage)
            prompt_text = f"{a}+{b}="
            prompt = TOK.encode(prompt_text, eos=False)
            p_ids  = torch.tensor([prompt], dtype=torch.long, device=device)

            # Run prompt to get post-prompt hidden state
            h = model.cell.init_h(1)
            for t in range(p_ids.size(1) - 1):
                _, h = model.cell.step(p_ids[0, t:t+1], h)
                h = cap_registers(h, R)
            post_prompt = h.clone()

            # Generate answer
            gen = TOK.decode(model.generate(p_ids))
            pred = _extract(gen, 'ans:')

            # Per-slot snapshot at end of prompt
            slots = [post_prompt[0, i*R:(i+1)*R].cpu().numpy() for i in range(6)]
            if pred == true_ans:
                correct_states.append(slots)
                correct_finals.append(post_prompt[0].cpu().numpy())
            else:
                wrong_states.append(slots)
                wrong_finals.append(post_prompt[0].cpu().numpy())

    n_c, n_w = len(correct_states), len(wrong_states)
    print(f"\n  stage {stage}: {n_c} correct, {n_w} wrong  ({n_c/(n_c+n_w):.1%} acc)")

    if n_w < 5:
        print(f"  too few wrong examples for stable comparison")
        return None
    if n_c < 5:
        print(f"  too few correct examples for stable comparison")
        return None

    # For each slot, compute mean and std for each group, plus the
    # normalized distance between the two means (Mahalanobis-like).
    slot_divergence = []
    for i in range(6):
        c = np.array([s[i] for s in correct_states])   # (n_c, R)
        w = np.array([s[i] for s in wrong_states])     # (n_w, R)
        mu_c, mu_w = c.mean(0), w.mean(0)
        # Pooled std for normalization
        pooled = np.concatenate([c, w]).std(0) + 1e-6
        d = np.linalg.norm((mu_c - mu_w) / pooled)
        slot_divergence.append(d)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    bars = ax.bar(REG_NAMES, slot_divergence, color=colors, alpha=0.8)
    ax.set_ylabel("normalized distance between\ncorrect & wrong post-prompt states")
    ax.set_title(f"Failure-conditioned register divergence | stage {stage} | n={n_problems}")
    ax.grid(alpha=0.3, axis='y')
    for b, d in zip(bars, slot_divergence):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                f"{d:.2f}", ha='center', fontsize=10)
    out = os.path.join(DIAG_DIR, f'failure_divergence_stage{stage}.png')
    plt.tight_layout(); plt.savefig(out, dpi=130); plt.close()
    print(f"  saved {out}")

    print(f"\n  Slot divergence (higher = slot more responsible for failure):")
    ranked = sorted(enumerate(slot_divergence), key=lambda x: -x[1])
    for rank, (i, d) in enumerate(ranked):
        print(f"    {rank+1}. {REG_NAMES[i]:8s} {d:.3f}")
    return slot_divergence


# ── 3. SLOT ABLATION ─────────────────────────────────────────────────────────
def slot_ablation(model, stage, n_problems=200):
    """
    For each slot, zero it out at every timestep during inference and
    measure the accuracy drop. The bigger the drop, the more load-bearing
    the slot.

    Reads: is the explicit register partitioning actually doing work?
    """
    R = REG_DIM

    def eval_with_ablation(slot_idx):
        """slot_idx in {0..5} or None for baseline."""
        model.eval()
        ok_ans = ok_carry = total = carry_total = 0
        with torch.no_grad():
            for _ in range(n_problems):
                text, a, b, true_ans, true_carry = make_problem(stage)
                prompt = TOK.encode(f"{a}+{b}=", eos=False)
                p_ids  = torch.tensor([prompt], dtype=torch.long, device=device)

                # Custom generation loop with per-step ablation
                h = model.cell.init_h(1)
                for t in range(p_ids.size(1) - 1):
                    _, h = model.cell.step(p_ids[0, t:t+1], h)
                    if slot_idx is not None:
                        h[:, slot_idx*R:(slot_idx+1)*R] = 0.0
                    h = cap_registers(h, R)

                out_ids = list(p_ids[0].tolist())
                x = p_ids[0, -1:]
                for _ in range(60):
                    lg, h = model.cell.step(x, h)
                    if slot_idx is not None:
                        h[:, slot_idx*R:(slot_idx+1)*R] = 0.0
                    h = cap_registers(h, R)
                    nx = lg.argmax(-1)
                    out_ids.append(nx.item())
                    if nx.item() == TOK.eos_id:
                        break
                    x = nx

                gen = TOK.decode(out_ids)
                if _extract(gen, 'ans:') == true_ans:
                    ok_ans += 1
                if stage in CARRY_STAGES:
                    carry_total += 1
                    if _extract(gen, 'carry:') == true_carry:
                        ok_carry += 1
                total += 1
        ans_acc   = ok_ans / total
        carry_acc = ok_carry / carry_total if carry_total else None
        return ans_acc, carry_acc

    # Baseline (no ablation)
    base_ans, base_carry = eval_with_ablation(None)
    print(f"\n  stage {stage} baseline: ans={base_ans:.3f}", end="")
    if base_carry is not None:
        print(f"  carry={base_carry:.3f}")
    else:
        print()

    drops_ans, drops_carry = [], []
    for i in range(6):
        a_acc, c_acc = eval_with_ablation(i)
        drops_ans.append(base_ans - a_acc)
        drops_carry.append((base_carry - c_acc) if c_acc is not None else 0.0)
        if c_acc is not None:
            print(f"    ablate {REG_NAMES[i]:8s} ans={a_acc:.3f} (Δ{drops_ans[-1]:+.3f}) "
                  f"carry={c_acc:.3f} (Δ{drops_carry[-1]:+.3f})")
        else:
            print(f"    ablate {REG_NAMES[i]:8s} ans={a_acc:.3f} (Δ{drops_ans[-1]:+.3f})")

    fig, axes = plt.subplots(1, 2 if base_carry is not None else 1,
                             figsize=(12 if base_carry is not None else 7, 5),
                             squeeze=False)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    ax = axes[0, 0]
    ax.bar(REG_NAMES, drops_ans, color=colors, alpha=0.8)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel("answer accuracy drop\n(higher = more load-bearing)")
    ax.set_title(f"Slot ablation: answer | stage {stage}")
    ax.grid(alpha=0.3, axis='y')

    if base_carry is not None:
        ax = axes[0, 1]
        ax.bar(REG_NAMES, drops_carry, color=colors, alpha=0.8)
        ax.axhline(0, color='k', lw=0.5)
        ax.set_ylabel("carry accuracy drop")
        ax.set_title(f"Slot ablation: carry | stage {stage}")
        ax.grid(alpha=0.3, axis='y')

    out = os.path.join(DIAG_DIR, f'slot_ablation_stage{stage}.png')
    plt.tight_layout(); plt.savefig(out, dpi=130); plt.close()
    print(f"  saved {out}")
    return drops_ans, drops_carry


# ── RUN ALL THREE ────────────────────────────────────────────────────────────
def run_diagnostics(model, stages=(2, 4)):
    """
    Run all three diagnostics on the given stages.
    Default: stage 2 (single-digit carry) and stage 4 (two-digit carry).
    These are the stages where the carry register should be doing real work.
    """
    print("=" * 60)
    print("VRU v23 REGISTER DIAGNOSTICS")
    print("=" * 60)
    for stage in stages:
        print(f"\n--- STAGE {stage}: {STAGES[stage][4]} ---")
        print("\n[1/3] Activation map...")
        activation_map(model, stage, n_problems=50)
        print("\n[2/3] Failure-conditioned divergence...")
        failure_conditioned(model, stage, n_problems=200)
        print("\n[3/3] Slot ablation...")
        slot_ablation(model, stage, n_problems=150)
    print(f"\nAll diagnostics saved to {DIAG_DIR}")


# Run it (model must be trained and in memory; or load a checkpoint first):
run_diagnostics(model, stages=(2, 4))

NameError: name 'DRIVE_DIR' is not defined

In [2]:
# ============================================================
# HIDDEN STATE PERTURBATION MAP
# Take the trained Lorenz agent. Free-roll it. Periodically kick
# its hidden state. Map where in state space it goes.
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa


# ---- Sim and agent (same as before) ----
class LorenzSim:
    def __init__(self, dt=0.01, seed=1):
        rng = np.random.RandomState(seed)
        self.dt = dt
        self.state = np.array([1.0, 1.0, 1.0]) + 0.01 * rng.randn(3)
    def step(self):
        x, y, z = self.state
        dx = 10.0 * (y - x)
        dy = x * (28.0 - z) - y
        dz = x * y - (8/3) * z
        self.state = self.state + self.dt * np.array([dx, dy, dz])
        return self.state.copy()


class PredictiveAgent(nn.Module):
    def __init__(self, obs_dim=3, hidden_dim=64):
        super().__init__()
        self.cell = nn.GRUCell(obs_dim, hidden_dim)
        self.predict = nn.Linear(hidden_dim, obs_dim)
        self.hidden_dim = hidden_dim
    def forward(self, obs, h):
        h = self.cell(obs, h)
        return self.predict(h), h
    def init_hidden(self, device):
        return torch.zeros(1, self.hidden_dim, device=device)


# ---- Train (quick — same as before, abbreviated) ----
def train(steps=15000, horizon=20, lr=1e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sim = LorenzSim()
    agent = PredictiveAgent().to(device)
    opt = torch.optim.Adam(agent.parameters(), lr=lr)
    burn = np.array([sim.step() for _ in range(2000)])
    mu, sd = burn.mean(0), burn.std(0)
    obs = torch.tensor((sim.step() - mu) / sd, dtype=torch.float32,
                       device=device).unsqueeze(0)
    h = agent.init_hidden(device)
    future = [torch.tensor((sim.step() - mu) / sd, dtype=torch.float32,
                           device=device).unsqueeze(0) for _ in range(horizon)]
    for step in range(steps):
        ih, io = h, obs
        for _ in range(horizon):
            ip, ih = agent(io, ih); io = ip
        loss = ((ip - future[-1]) ** 2).mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(agent.parameters(), 1.0)
        opt.step()
        nxt = future.pop(0)
        future.append(torch.tensor((sim.step() - mu) / sd, dtype=torch.float32,
                                   device=device).unsqueeze(0))
        with torch.no_grad():
            _, h = agent(obs, h)
        obs = nxt.detach(); h = h.detach()
    print(f"Training done. Final loss: {loss.item():.4f}")
    return agent, mu, sd, device


# ---- The experiment ----
def perturbation_map(agent, mu, sd, device,
                     baseline_steps=3000,
                     kick_interval=80,    # kick every N steps
                     kick_magnitudes=[0.5, 1.0, 2.0, 5.0],
                     n_recover_steps=40,  # how long to watch after a kick
                     seed=99):
    """
    Run free rollout. Periodically kick the hidden state with a random
    direction at varying magnitudes. Record:
      - hidden state trajectory
      - prediction error trajectory
      - kick events (when, magnitude, direction)
      - "did it recover to baseline behavior"
    """
    sim = LorenzSim(seed=seed)
    for _ in range(2000): sim.step()

    agent.eval()
    h = agent.init_hidden(device)
    last = None
    # Warm up on real obs
    for _ in range(500):
        s = sim.step()
        o = torch.tensor((s - mu) / sd, dtype=torch.float32,
                         device=device).unsqueeze(0)
        with torch.no_grad():
            _, h = agent(o, h)
        last = o

    rng = np.random.RandomState(seed)
    history = []   # list of dicts per step
    cur = last
    next_kick = kick_interval
    pending_recover = None  # tracks state right after a kick

    with torch.no_grad():
        for step in range(baseline_steps):
            kick_info = None
            # Should we kick?
            if step == next_kick:
                mag = kick_magnitudes[rng.randint(len(kick_magnitudes))]
                direction = torch.randn_like(h)
                direction = direction / direction.norm() * mag
                h_pre = h.clone()
                h = h + direction
                kick_info = {
                    'step': step,
                    'magnitude': mag,
                    'direction': direction.cpu().numpy().squeeze(),
                    'h_pre': h_pre.cpu().numpy().squeeze(),
                    'h_post': h.cpu().numpy().squeeze(),
                }
                pending_recover = {
                    'kick_step': step,
                    'magnitude': mag,
                    'errors': [],
                    'h_traj': [],
                }
                next_kick = step + kick_interval

            cur, h = agent(cur, h)

            # Record
            pred = cur.cpu().numpy().squeeze() * sd + mu
            history.append({
                'step': step,
                'h': h.cpu().numpy().squeeze().copy(),
                'pred': pred,
                'kick': kick_info,
            })

            if pending_recover is not None:
                pending_recover['h_traj'].append(h.cpu().numpy().squeeze())
                if step - pending_recover['kick_step'] >= n_recover_steps:
                    history[-1]['recovery'] = pending_recover
                    pending_recover = None

    return history


def analyze_and_plot(history, agent, mu, sd, device, out_path):
    H = np.array([rec['h'] for rec in history])    # (T, 64)
    preds = np.array([rec['pred'] for rec in history])  # (T, 3)
    kicks = [rec for rec in history if rec['kick'] is not None]

    # PCA on full hidden-state trajectory
    Hc = H - H.mean(0)
    U, S, Vt = np.linalg.svd(Hc, full_matrices=False)
    H_pca = Hc @ Vt.T[:, :3]
    var3 = (S[:3]**2).sum() / (S**2).sum()

    # Build a baseline (no-kick) reference trajectory for comparison
    sim = LorenzSim(seed=999)
    for _ in range(2000): sim.step()
    h_ref = agent.init_hidden(device)
    last = None
    with torch.no_grad():
        for _ in range(500):
            s = sim.step()
            o = torch.tensor((s - mu) / sd, dtype=torch.float32,
                             device=device).unsqueeze(0)
            _, h_ref = agent(o, h_ref)
            last = o
        H_ref = []
        cur = last
        for _ in range(len(history)):
            cur, h_ref = agent(cur, h_ref)
            H_ref.append(h_ref.cpu().numpy().squeeze())
    H_ref = np.array(H_ref)
    H_ref_pca = (H_ref - H.mean(0)) @ Vt.T[:, :3]

    # Color hidden-state path by "time since last kick" so we can see recovery
    time_since_kick = np.full(len(history), 1e9)
    last_kick_step = -1e9
    for i, rec in enumerate(history):
        if rec['kick'] is not None:
            last_kick_step = i
        time_since_kick[i] = i - last_kick_step

    # ---- Plot ----
    fig = plt.figure(figsize=(16, 11))

    # 1. Hidden state PCA, kicked run, colored by time-since-kick
    ax = fig.add_subplot(2, 3, 1, projection='3d')
    sc = ax.scatter(H_pca[:, 0], H_pca[:, 1], H_pca[:, 2],
                    c=np.clip(time_since_kick, 0, 80),
                    cmap='plasma_r', s=2, alpha=0.6)
    ax.set_title(f"Hidden state (kicked run)\ncolor = steps since last kick "
                 f"| top-3 PC var: {var3:.0%}")
    plt.colorbar(sc, ax=ax, shrink=0.6)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")

    # 2. Hidden state PCA, baseline (no kicks)
    ax = fig.add_subplot(2, 3, 2, projection='3d')
    ax.scatter(H_ref_pca[:, 0], H_ref_pca[:, 1], H_ref_pca[:, 2],
               c='C0', s=2, alpha=0.5)
    ax.set_title("Hidden state (baseline, no kicks)\nsame PCA basis")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")

    # 3. Distance from kicked trajectory to baseline manifold
    # For each kicked-run point, find min distance to baseline-run point set
    from scipy.spatial import cKDTree
    tree = cKDTree(H_ref)
    dists, _ = tree.query(H, k=1)
    ax = fig.add_subplot(2, 3, 3)
    ax.plot(dists, color='C2', lw=0.7)
    for k in kicks:
        ax.axvline(k['step'], color='red', alpha=0.4, lw=0.6)
    ax.set_xlabel("step")
    ax.set_ylabel("distance to baseline manifold (hidden-state space)")
    ax.set_title("Drift after kicks (red lines = kick events)")
    ax.grid(alpha=0.3)

    # 4. Output prediction trajectory in obs space, kick locations marked
    ax = fig.add_subplot(2, 3, 4, projection='3d')
    ax.plot(preds[:, 0], preds[:, 1], preds[:, 2], lw=0.4, color='C2')
    kick_preds = np.array([preds[k['step']] for k in kicks])
    if len(kick_preds):
        ax.scatter(kick_preds[:, 0], kick_preds[:, 1], kick_preds[:, 2],
                   c='red', s=30, zorder=5, label='kick events')
        ax.legend()
    ax.set_title("Output predictions (Lorenz space)\nred = where kicks occurred")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

    # 5. Recovery curves: error post-kick, grouped by kick magnitude
    ax = fig.add_subplot(2, 3, 5)
    recoveries = [rec['recovery'] for rec in history if 'recovery' in rec]
    by_mag = {}
    for rec in recoveries:
        h_traj = np.array(rec['h_traj'])
        # Distance from baseline manifold over recovery window
        d = tree.query(h_traj, k=1)[0]
        mag = rec['magnitude']
        by_mag.setdefault(mag, []).append(d)
    cmap = plt.cm.viridis
    mags_sorted = sorted(by_mag.keys())
    for i, mag in enumerate(mags_sorted):
        traces = np.array(by_mag[mag])
        mean = traces.mean(0); std = traces.std(0)
        c = cmap(i / max(len(mags_sorted)-1, 1))
        ax.plot(mean, color=c, lw=2, label=f"kick mag = {mag}")
        ax.fill_between(np.arange(len(mean)), mean-std, mean+std,
                        color=c, alpha=0.2)
    ax.set_xlabel("steps after kick")
    ax.set_ylabel("distance to baseline manifold")
    ax.set_title("Recovery curves by kick magnitude")
    ax.legend()
    ax.grid(alpha=0.3)

    # 6. PCA scatter colored by kick magnitude (only post-kick points)
    ax = fig.add_subplot(2, 3, 6, projection='3d')
    # Plot baseline as gray reference
    ax.scatter(H_ref_pca[:, 0], H_ref_pca[:, 1], H_ref_pca[:, 2],
               c='lightgray', s=1, alpha=0.3)
    # Plot post-kick recovery trajectories
    for rec in recoveries:
        h_traj = np.array(rec['h_traj'])
        h_pca = (h_traj - H.mean(0)) @ Vt.T[:, :3]
        c = cmap(mags_sorted.index(rec['magnitude']) /
                 max(len(mags_sorted)-1, 1))
        ax.plot(h_pca[:, 0], h_pca[:, 1], h_pca[:, 2],
                color=c, lw=0.8, alpha=0.7)
    ax.set_title("Recovery trajectories\noverlaid on baseline (gray)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")

    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {out_path}")

    # ---- Print summary ----
    print(f"\n{'='*60}")
    print(f"PERTURBATION MAP SUMMARY")
    print(f"{'='*60}")
    print(f"Total steps:    {len(history)}")
    print(f"Total kicks:    {len(kicks)}")
    print(f"Recoveries:     {len(recoveries)}")
    print(f"\nMean drift from baseline manifold:")
    print(f"  pre-kick windows:  {dists[:200].mean():.3f}")
    print(f"  full kicked run:   {dists.mean():.3f}")
    print(f"\nRecovery by kick magnitude:")
    for mag in mags_sorted:
        traces = np.array(by_mag[mag])
        recovered = (traces[:, -1] < traces[:, 0] * 0.5).mean() * 100
        print(f"  mag {mag:4.1f}:  initial drift {traces[:, 0].mean():.2f}  "
              f"final drift {traces[:, -1].mean():.2f}  "
              f"recovered {recovered:.0f}% of trials")


# ---- Run ----
print("Training agent...")
agent, mu, sd, device = train()

print("\nRunning perturbation experiment...")
history = perturbation_map(agent, mu, sd, device)

print("\nAnalyzing...")
analyze_and_plot(history, agent, mu, sd, device, '/content/perturbation_map.png')

Training agent...
Training done. Final loss: 0.0587

Running perturbation experiment...

Analyzing...

Saved: /content/perturbation_map.png

PERTURBATION MAP SUMMARY
Total steps:    3000
Total kicks:    37
Recoveries:     36

Mean drift from baseline manifold:
  pre-kick windows:  1.279
  full kicked run:   1.444

Recovery by kick magnitude:
  mag  0.5:  initial drift 0.87  final drift 0.63  recovered 0% of trials
  mag  1.0:  initial drift 1.61  final drift 1.11  recovered 11% of trials
  mag  2.0:  initial drift 2.05  final drift 1.35  recovered 20% of trials
  mag  5.0:  initial drift 4.13  final drift 2.30  recovered 36% of trials
